<a href="https://colab.research.google.com/github/RozieKii/localTest/blob/main/10_tool_use_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [8]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    """Calculate math expressions."""
    try:
        result = eval(expression)  # Unsafe for untrusted input
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}


# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    """Get current time."""
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }


print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '19:09:26', 'date': '2026-10-28', 'day': 'Wednesday'}


## Part 2: Tool Registry

Manage available tools.

In [4]:
class ToolRegistry:
    """Manages available tools for the agent."""

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        """Register a tool."""
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        """Get tool function by name."""
        return self.tools[name]['function']

    def list_tools(self) -> str:
        """Get formatted list of available tools."""
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)


# Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


In [5]:
calc_tool = registry.get_tool('calculator')
print(calc_tool('15 * 0.15'))

{'result': 2.25, 'expression': '15 * 0.15'}


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [10]:

class SimpleReActAgent:
    """Simple agent that can use tools."""

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        query_lower = query.lower()

        if any(word in query_lower for word in
               ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in
                 ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        if action == 'use_calculator':
            import re
            match = re.search(
                r'\d[\d\s.]*\s*[+\-*/]\s*[\d\s.]+',
                query
            )

            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)

            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        thought = self.think(query)
        self.history.append({
            'step': 'thought',
            'content': thought
        })

        observation = self.act(thought, query)
        self.history.append({
            'step': 'action',
            'content': observation
        })

        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = (
                f"It's {observation['time']} on "
                f"{observation['date']} ({observation['day']})"
            )
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer


# Create a fresh agent using your existing registry
agent = SimpleReActAgent(registry)

print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))

print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Q: What is 25 * 4?
A: The result is 100

Q: What time is it?
A: It's 19:10:47 on 2026-10-28 (Wednesday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [11]:

class DigitalTwinWithTools:
    """Digital twin that can use tools."""

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        """Setup default tools."""
        self.registry.register(
            'calculator', calculator, 'Calculate math'
        )
        self.registry.register(
            'get_time', get_time, 'Get current time'
        )
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        """Add a new tool."""
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        """Chat with tool use."""
        return self.agent.run(message)


# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))



Digital Twin with Tools:
The result is 7.125
It's 19:14:28 on 2026-10-28 (Wednesday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [12]:

# Custom Tool 1: Note taker
def save_note(arg: str) -> Dict[str, Any]:
    """Save a note in the digital twin's session memory."""
    if not isinstance(arg, str) or not arg.strip():
        return {'error': 'Note cannot be empty'}

    if not hasattr(twin, 'notes'):
        twin.notes = []

    note = arg.strip()
    twin.notes.append(note)

    return {
        'result': 'Note saved successfully',
        'note': note,
        'total_notes': len(twin.notes)
    }


# Custom Tool 2: Task list
def add_task(arg: str) -> Dict[str, Any]:
    """Add a task to the digital twin's session task list."""
    if not isinstance(arg, str) or not arg.strip():
        return {'error': 'Task cannot be empty'}

    if not hasattr(twin, 'tasks'):
        twin.tasks = []

    task = arg.strip()
    twin.tasks.append({
        'task': task,
        'completed': False
    })

    return {
        'result': 'Task added successfully',
        'task': task,
        'total_tasks': len(twin.tasks)
    }


# Register both tools
twin.add_tool('save_note', save_note, 'Save a note')
twin.add_tool('add_task', add_task, 'Add a task to the task list')

# Test the tools directly
print(twin.registry.get_tool('save_note')(
    'Review my AI agent notes'
))

print(twin.registry.get_tool('add_task')(
    'Complete Week 3.1 exercises'
))


{'result': 'Note saved successfully', 'note': 'Review my AI agent notes', 'total_notes': 1}
{'result': 'Task added successfully', 'task': 'Complete Week 3.1 exercises', 'total_tasks': 1}


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI